# Capstone Phase 1: Preprocessing & Feature Engineering Pipeline
**Project:** Intelligent Phishing Website Detection Using Machine Learning & Explainable AI  
**Objective:** Build a leak-free, stratified train-test split pipeline with feature scaling and importance ranking.

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load raw dataset
raw_df = pd.read_csv('../data/raw/phishing_dataset.csv')
print('Raw shape:', raw_df.shape)
raw_df.head(3)

In [ ]:
# Remap target {-1, 1} -> {0: Legitimate, 1: Phishing}
X = raw_df.drop(columns=['Result'])
y = raw_df['Result'].map({-1: 0, 1: 1}).astype(int)

# Stratified 80/20 train/test split with random_state=42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')
print(f'Train phishing ratio: {y_train.mean():.4f} | Test phishing ratio: {y_test.mean():.4f}')

In [ ]:
# Zero-Leakage Preprocessing
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit ONLY on train
X_test_scaled = scaler.transform(X_test)        # Transform ONLY on test
print('Scaling completed without leakage!')

In [ ]:
# Feature Selection: Mutual Information
from sklearn.feature_selection import mutual_info_classif
mi = mutual_info_classif(X_train, y_train, random_state=42)
mi_df = pd.DataFrame({'feature': X.columns, 'mi_score': mi}).sort_values('mi_score', ascending=False)
mi_df.head(10)